# 📉 Retail Sales Decline Analysis: 2016 → 2017
## Storytelling: Why Did Sales Drop?

**Objective:** Deep dive analysis to uncover the root causes of the 1.4% sales decline from 2016 to 2017

---
### Executive Summary

- **2016 Total Sales:** $9.32M (Peak)
- **2017 Total Sales:** $9.18M
- **Decline:** -$134,648 (-1.4%)
- **Order Count Change:** -472 orders (-1.8%)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import json
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)

# Database Connection
with open('config/db_config.json') as config_file:
    config = json.load(config_file)

DB_URI = f"postgresql+psycopg2://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['dbname']}?options=-csearch_path=dw"
engine = create_engine(DB_URI)

print("✅ Database connected successfully!")

## 1️⃣ High-Level View: The Story Begins

In [ ]:
# Year-over-Year Sales Comparison
query = """
SELECT 
    dc.year,
    SUM(fs.totalsales)::NUMERIC AS total_sales,
    COUNT(fs.salesid)::INT AS order_count,
    AVG(fs.totalsales)::NUMERIC AS avg_order_value,
    COUNT(DISTINCT fs.customerid)::INT AS unique_customers
FROM dw.fact_sales fs
JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
WHERE dc.year BETWEEN 2015 AND 2017
GROUP BY dc.year
ORDER BY dc.year
"""

yearly_data = pd.read_sql_query(query, engine)
print("\n📊 YEARLY PERFORMANCE METRICS:")
print("="*70)
print(yearly_data.to_string(index=False))

# Calculate YoY growth
print("\n📈 YEAR-OVER-YEAR GROWTH:")
print("="*70)
for i in range(1, len(yearly_data)):
    prev_year = yearly_data.iloc[i-1]
    curr_year = yearly_data.iloc[i]
    
    sales_growth = ((curr_year['total_sales'] - prev_year['total_sales']) / prev_year['total_sales'] * 100)
    order_growth = ((curr_year['order_count'] - prev_year['order_count']) / prev_year['order_count'] * 100)
    aov_growth = ((curr_year['avg_order_value'] - prev_year['avg_order_value']) / prev_year['avg_order_value'] * 100)
    
    print(f"\n{int(prev_year['year'])} → {int(curr_year['year'])}:")
    print(f"  Sales Growth: {sales_growth:+.2f}%")
    print(f"  Order Count Growth: {order_growth:+.2f}%")
    print(f"  AVG Order Value Growth: {aov_growth:+.2f}%")
    print(f"  Customer Growth: {((curr_year['unique_customers'] - prev_year['unique_customers']) / prev_year['unique_customers'] * 100):+.2f}%")

In [ ]:
# Visual: Yearly Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Year-over-Year Performance Comparison (2015-2017)', fontsize=16, fontweight='bold')

# Sales
axes[0, 0].bar(yearly_data['year'], yearly_data['total_sales']/1e6, color=['#3498db', '#e74c3c', '#f39c12'])
axes[0, 0].set_title('Total Sales By Year', fontweight='bold')
axes[0, 0].set_ylabel('Sales (Millions $)')
for i, v in enumerate(yearly_data['total_sales']/1e6):
    axes[0, 0].text(yearly_data['year'].iloc[i], v + 0.1, f'${v:.2f}M', ha='center', fontweight='bold')

# Orders
axes[0, 1].plot(yearly_data['year'], yearly_data['order_count'], marker='o', linewidth=2, markersize=10, color='#2ecc71')
axes[0, 1].set_title('Total Orders By Year', fontweight='bold')
axes[0, 1].set_ylabel('Number of Orders')
axes[0, 1].fill_between(yearly_data['year'], yearly_data['order_count'], alpha=0.3, color='#2ecc71')

# Average Order Value
axes[1, 0].plot(yearly_data['year'], yearly_data['avg_order_value'], marker='s', linewidth=2, markersize=10, color='#9b59b6')
axes[1, 0].set_title('Average Order Value Trend', fontweight='bold')
axes[1, 0].set_ylabel('Average Order Value ($)')
axes[1, 0].fill_between(yearly_data['year'], yearly_data['avg_order_value'], alpha=0.3, color='#9b59b6')

# Customers
axes[1, 1].bar(yearly_data['year'], yearly_data['unique_customers'], color=['#1abc9c', '#3498db', '#34495e'])
axes[1, 1].set_title('Active Customers By Year', fontweight='bold')
axes[1, 1].set_ylabel('Number of Customers')

plt.tight_layout()
plt.savefig('reports/yearly_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Chart saved: reports/yearly_comparison.png")

## 2️⃣ Monthly Deep Dive: When Did the Decline Start?

In [ ]:
# Monthly comparison
query = """
SELECT 
    dc.year,
    dc.month,
    SUM(fs.totalsales)::NUMERIC AS total_sales,
    COUNT(fs.salesid)::INT AS order_count,
    AVG(fs.totalsales)::NUMERIC AS avg_order_value
FROM dw.fact_sales fs
JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
WHERE dc.year IN (2016, 2017)
GROUP BY dc.year, dc.month
ORDER BY dc.year, dc.month
"""

monthly_comp = pd.read_sql_query(query, engine)

# Pivot for comparison
sales_pivot = monthly_comp.pivot(index='month', columns='year', values='total_sales')
orders_pivot = monthly_comp.pivot(index='month', columns='year', values='order_count')

# Calculate month-by-month decline
sales_pivot['change_$'] = sales_pivot[2017] - sales_pivot[2016]
sales_pivot['change_%'] = (sales_pivot[2017] - sales_pivot[2016]) / sales_pivot[2016] * 100

print("\n📅 MONTHLY SALES COMPARISON (2016 vs 2017):")
print("="*80)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for idx, row in sales_pivot.iterrows():
    print(f"\n{month_names[int(idx)-1]:>3} | 2016: ${row[2016]:>12,.0f} | 2017: ${row[2017]:>12,.0f} | Change: ${row['change_$']:>10,.0f} ({row['change_%']:>6.1f}%)")

print(f"\n\n{'TOTAL':>3} | 2016: ${sales_pivot[2016].sum():>12,.0f} | 2017: ${sales_pivot[2017].sum():>12,.0f} | Change: ${sales_pivot['change_$'].sum():>10,.0f} ({sales_pivot['change_%'].mean():>6.1f}%)")

In [ ]:
# Visualize monthly trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('2016 vs 2017: Monthly Sales Trends', fontsize=16, fontweight='bold')

# Side-by-side monthly comparison
x = np.arange(len(month_names))
width = 0.35

axes[0].bar(x - width/2, sales_pivot[2016], width, label='2016', color='#3498db', alpha=0.8)
axes[0].bar(x + width/2, sales_pivot[2017], width, label='2017', color='#e74c3c', alpha=0.8)
axes[0].set_ylabel('Total Sales ($)', fontweight='bold')
axes[0].set_title('Monthly Sales Comparison', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(month_names)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Month-by-month change
colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in sales_pivot['change_%']]
axes[1].bar(month_names, sales_pivot['change_%'], color=colors, alpha=0.8)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].set_ylabel('% Change 2016→2017', fontweight='bold')
axes[1].set_title('Month-by-Month Change (%)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(sales_pivot['change_%']):
    axes[1].text(i, v + (1 if v > 0 else -1), f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('reports/monthly_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Chart saved: reports/monthly_comparison.png")

## 3️⃣ Product Analysis: Which Products Hurt Performance?

In [ ]:
# Product-level decline analysis
query = """
SELECT 
    dp.productname,
    SUM(CASE WHEN dc.year = 2016 THEN fs.totalsales ELSE 0 END)::NUMERIC AS sales_2016,
    SUM(CASE WHEN dc.year = 2017 THEN fs.totalsales ELSE 0 END)::NUMERIC AS sales_2017,
    COUNT(CASE WHEN dc.year = 2016 THEN fs.salesid END)::INT AS orders_2016,
    COUNT(CASE WHEN dc.year = 2017 THEN fs.salesid END)::INT AS orders_2017,
    AVG(CASE WHEN dc.year = 2016 THEN fs.totalsales ELSE NULL END)::NUMERIC AS avg_2016,
    AVG(CASE WHEN dc.year = 2017 THEN fs.totalsales ELSE NULL END)::NUMERIC AS avg_2017
FROM dw.fact_sales fs
JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
JOIN dw.dim_products dp ON fs.productid = dp.productid
WHERE dc.year IN (2016, 2017)
GROUP BY dp.productname
ORDER BY (sales_2016 - sales_2017) DESC
"""

product_analysis = pd.read_sql_query(query, engine)
product_analysis['sales_change'] = product_analysis['sales_2017'] - product_analysis['sales_2016']
product_analysis['change_%'] = (product_analysis['sales_change'] / product_analysis['sales_2016'] * 100).round(2)
product_analysis['order_change'] = product_analysis['orders_2017'] - product_analysis['orders_2016']

print("\n🏆 TOP 5 PRODUCTS THAT DECLINED:")
print("="*100)
top_declining = product_analysis.head(5)
for idx, row in top_declining.iterrows():
    print(f"\nProduct: {row['productname']}")
    print(f"  2016 Sales: ${row['sales_2016']:,.0f} ({int(row['orders_2016'])} orders)")
    print(f"  2017 Sales: ${row['sales_2017']:,.0f} ({int(row['orders_2017'])} orders)")
    print(f"  Change: ${row['sales_change']:,.0f} ({row['change_%']:+.1f}%)")
    print(f"  Order Decline: {int(row['order_change'])} orders")

print("\n\n🚀 TOP 5 PRODUCTS THAT GREW:")
print("="*100)
top_growing = product_analysis.tail(5).sort_values('sales_change', ascending=False)
for idx, row in top_growing.iterrows():
    print(f"\nProduct: {row['productname']}")
    print(f"  2016 Sales: ${row['sales_2016']:,.0f} ({int(row['orders_2016'])} orders)")
    print(f"  2017 Sales: ${row['sales_2017']:,.0f} ({int(row['orders_2017'])} orders)")
    print(f"  Change: ${row['sales_change']:,.0f} ({row['change_%']:+.1f}%)")
    print(f"  Order Growth: {int(row['order_change'])} orders")

In [ ]:
# Product visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Product Performance: 2016 vs 2017', fontsize=16, fontweight='bold')

# Top declining products
declining = product_analysis.nlargest(10, 'sales_change', keep='first')
declining = declining.sort_values('sales_change')

axes[0].barh(declining['productname'].str[:30], declining['sales_change']/1000, color='#e74c3c')
axes[0].set_xlabel('Sales Change ($1000s)', fontweight='bold')
axes[0].set_title('Top 10 Products with Biggest Decline', fontweight='bold')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.8)

# Top growing products
growing = product_analysis.nsmallest(10, 'sales_change', keep='first')
growing = growing.sort_values('sales_change', ascending=False)

axes[1].barh(growing['productname'].str[:30], growing['sales_change']/1000, color='#2ecc71')
axes[1].set_xlabel('Sales Change ($1000s)', fontweight='bold')
axes[1].set_title('Top 10 Products with Biggest Growth', fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.8)

plt.tight_layout()
plt.savefig('reports/product_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Chart saved: reports/product_analysis.png")

## 4️⃣ Regional Analysis: Geographic Impact

In [ ]:
# Regional analysis
query = """
SELECT 
    dt.territoryname,
    SUM(CASE WHEN dc.year = 2016 THEN fs.totalsales ELSE 0 END)::NUMERIC AS sales_2016,
    SUM(CASE WHEN dc.year = 2017 THEN fs.totalsales ELSE 0 END)::NUMERIC AS sales_2017,
    COUNT(CASE WHEN dc.year = 2016 THEN fs.salesid END)::INT AS orders_2016,
    COUNT(CASE WHEN dc.year = 2017 THEN fs.salesid END)::INT AS orders_2017
FROM dw.fact_sales fs
JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
JOIN dw.dim_territories dt ON fs.territoryid = dt.territoryid
WHERE dc.year IN (2016, 2017)
GROUP BY dt.territoryname
ORDER BY sales_2016 DESC
"""

regional_analysis = pd.read_sql_query(query, engine)
regional_analysis['sales_change'] = regional_analysis['sales_2017'] - regional_analysis['sales_2016']
regional_analysis['change_%'] = (regional_analysis['sales_change'] / regional_analysis['sales_2016'] * 100).round(2)
regional_analysis['order_change'] = regional_analysis['orders_2017'] - regional_analysis['orders_2016']

print("\n🌍 REGIONAL PERFORMANCE: 2016 vs 2017")
print("="*100)
for idx, row in regional_analysis.iterrows():
    change_indicator = '📈' if row['sales_change'] > 0 else '📉'
    print(f"\n{change_indicator} {row['territoryname']}")
    print(f"  2016: ${row['sales_2016']:>12,.0f} | Orders: {int(row['orders_2016']):>5}")
    print(f"  2017: ${row['sales_2017']:>12,.0f} | Orders: {int(row['orders_2017']):>5}")
    print(f"  Change: ${row['sales_change']:>12,.0f} ({row['change_%']:>6.1f}%) | Orders: {int(row['order_change']):>+5}")

# Regional summary
worst_region = regional_analysis.loc[regional_analysis['sales_change'].idxmin()]
best_region = regional_analysis.loc[regional_analysis['sales_change'].idxmax()]

print(f"\n\n📉 BIGGEST LOOSER: {worst_region['territoryname']} ({worst_region['change_%']:+.1f}%)")
print(f"📈 BEST PERFORMER: {best_region['territoryname']} ({best_region['change_%']:+.1f}%)")

In [ ]:
# Regional visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Regional Performance: 2016 vs 2017', fontsize=16, fontweight='bold')

# Side-by-side comparison
x = np.arange(len(regional_analysis))
width = 0.35

axes[0].barh(x - width/2, regional_analysis['sales_2016']/1e6, width, label='2016', color='#3498db')
axes[0].barh(x + width/2, regional_analysis['sales_2017']/1e6, width, label='2017', color='#e74c3c')
axes[0].set_yticks(x)
axes[0].set_yticklabels(regional_analysis['territoryname'])
axes[0].set_xlabel('Sales (Millions $)', fontweight='bold')
axes[0].set_title('Territory Sales Comparison', fontweight='bold')
axes[0].legend()

# Percentage change
colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in regional_analysis['change_%']]
axes[1].barh(regional_analysis['territoryname'], regional_analysis['change_%'], color=colors)
axes[1].set_xlabel('% Change 2016→2017', fontweight='bold')
axes[1].set_title('Regional Sales Change (%)', fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.8)

for i, v in enumerate(regional_analysis['change_%']):
    axes[1].text(v + (0.2 if v > 0 else -0.2), i, f'{v:.1f}%', va='center', ha='left' if v > 0 else 'right', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/regional_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Chart saved: reports/regional_analysis.png")

## 5️⃣ Customer Behavior Analysis: Are Customers Buying Less?

In [ ]:
# Customer metrics
query = """
SELECT 
    dc.year,
    COUNT(DISTINCT fs.customerid)::INT AS unique_customers,
    COUNT(DISTINCT fs.customerid) FILTER (WHERE fs.totalsales > 0)::INT AS active_customers,
    COUNT(fs.salesid)::INT AS total_transactions,
    AVG(fs.totalsales)::NUMERIC AS avg_transaction_value,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY fs.totalsales)::NUMERIC AS median_transaction,
    SUM(fs.quantity)::INT AS total_quantity,
    AVG(fs.quantity)::NUMERIC AS avg_quantity_per_order
FROM dw.fact_sales fs
JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
WHERE dc.year IN (2016, 2017)
GROUP BY dc.year
ORDER BY dc.year
"""

customer_behavior = pd.read_sql_query(query, engine)

print("\n👥 CUSTOMER BEHAVIOR METRICS:")
print("="*100)

for idx, row in customer_behavior.iterrows():
    print(f"\n📅 {int(row['year'])}:")
    print(f"  Unique Customers: {int(row['unique_customers']):,}")
    print(f"  Active Customers: {int(row['active_customers']):,}")
    print(f"  Total Transactions: {int(row['total_transactions']):,}")
    print(f"  Avg Transaction Value: ${row['avg_transaction_value']:.2f}")
    print(f"  Median Transaction Value: ${row['median_transaction']:.2f}")
    print(f"  Total Quantity Sold: {int(row['total_quantity']):,} units")
    print(f"  Avg Quantity per Order: {row['avg_quantity_per_order']:.2f} units")

# Year-over-year changes
if len(customer_behavior) == 2:
    row_2016 = customer_behavior.iloc[0]
    row_2017 = customer_behavior.iloc[1]
    
    print("\n\n📊 2016 → 2017 CHANGES:")
    print("="*100)
    print(f"Customer Growth: {int(row_2017['unique_customers'] - row_2016['unique_customers']):+,} ({(row_2017['unique_customers'] - row_2016['unique_customers'])/row_2016['unique_customers']*100:+.1f}%)")
    print(f"Transaction Count Change: {int(row_2017['total_transactions'] - row_2016['total_transactions']):+,} ({(row_2017['total_transactions'] - row_2016['total_transactions'])/row_2016['total_transactions']*100:+.1f}%)")
    print(f"Avg Transaction Value Change: ${row_2017['avg_transaction_value'] - row_2016['avg_transaction_value']:+.2f} ({(row_2017['avg_transaction_value'] - row_2016['avg_transaction_value'])/row_2016['avg_transaction_value']*100:+.1f}%)")
    print(f"Quantity Sold Change: {int(row_2017['total_quantity'] - row_2016['total_quantity']):+,} units ({(row_2017['total_quantity'] - row_2016['total_quantity'])/row_2016['total_quantity']*100:+.1f}%)")
    print(f"Avg Quantity per Order Change: {row_2017['avg_quantity_per_order'] - row_2016['avg_quantity_per_order']:+.2f} units")

In [ ]:
# Analyze repeat customers
query = """
WITH customer_orders AS (
    SELECT 
        fs.customerid,
        dc.year,
        COUNT(fs.salesid)::INT AS order_count,
        SUM(fs.totalsales)::NUMERIC AS total_spent
    FROM dw.fact_sales fs
    JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
    WHERE dc.year IN (2016, 2017)
    GROUP BY fs.customerid, dc.year
)
SELECT 
    dc.year,
    CASE 
        WHEN order_count = 1 THEN '1 Order'
        WHEN order_count BETWEEN 2 AND 5 THEN '2-5 Orders'
        WHEN order_count BETWEEN 6 AND 10 THEN '6-10 Orders'
        ELSE '10+ Orders'
    END AS customer_segment,
    COUNT(DISTINCT customerid)::INT AS customer_count,
    AVG(total_spent)::NUMERIC AS avg_lifetime_value
FROM customer_orders
WHERE order_count >= 1
GROUP BY dc.year, customer_segment
ORDER BY dc.year
"""

customer_segments = pd.read_sql_query(query, engine)

print("\n\n🎯 CUSTOMER SEGMENTATION BY PURCHASE FREQUENCY:")
print("="*100)
print(customer_segments.to_string(index=False))

## 6️⃣ Price & Quantity Analysis: Price Elasticity?

In [ ]:
# Price and quantity analysis
query = """
SELECT 
    dc.year,
    SUM(fs.quantity)::INT AS total_quantity,
    AVG(fs.unitprice)::NUMERIC AS avg_unit_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY fs.unitprice)::NUMERIC AS median_unit_price,
    AVG(fs.totalsales/NULLIF(fs.quantity, 0))::NUMERIC AS avg_effective_price,
    SUM(fs.totalsales)::NUMERIC AS total_revenue
FROM dw.fact_sales fs
JOIN dw.dim_calendar dc ON fs.dateid = dc.dateid
WHERE dc.year IN (2016, 2017)
GROUP BY dc.year
ORDER BY dc.year
"""

price_analysis = pd.read_sql_query(query, engine)

print("\n💰 PRICE & QUANTITY ANALYSIS:")
print("="*100)

for idx, row in price_analysis.iterrows():
    print(f"\n📅 {int(row['year'])}:")
    print(f"  Total Quantity Sold: {int(row['total_quantity']):,} units")
    print(f"  Avg Unit Price: ${row['avg_unit_price']:.2f}")
    print(f"  Median Unit Price: ${row['median_unit_price']:.2f}")
    print(f"  Avg Effective Price (per unit sold): ${row['avg_effective_price']:.2f}")
    print(f"  Total Revenue: ${row['total_revenue']:,.0f}")

# Calculate changes
if len(price_analysis) == 2:
    row_2016 = price_analysis.iloc[0]
    row_2017 = price_analysis.iloc[1]
    
    qty_change = ((row_2017['total_quantity'] - row_2016['total_quantity']) / row_2016['total_quantity'] * 100)
    price_change = ((row_2017['avg_unit_price'] - row_2016['avg_unit_price']) / row_2016['avg_unit_price'] * 100)
    
    print("\n\n📊 2016 → 2017 CHANGES:")
    print("="*100)
    print(f"Quantity Change: {int(row_2017['total_quantity'] - row_2016['total_quantity']):+,} units ({qty_change:+.1f}%)")
    print(f"Avg Unit Price Change: ${row_2017['avg_unit_price'] - row_2016['avg_unit_price']:+.2f} ({price_change:+.1f}%)")
    print(f"\n💡 INSIGHT: Sales declined primarily due to {'LOWER QUANTITIES' if qty_change < 0 else 'LOWER PRICES'} (Qty: {qty_change:+.1f}% | Price: {price_change:+.1f}%)")

## 7️⃣ Key Findings & Root Causes

In [ ]:
# Summary findings
print("\n" + "="*100)
print("🔍 ROOT CAUSE ANALYSIS: WHY SALES DECLINED FROM 2016 → 2017")
print("="*100)

# Calculate key metrics
yearly = yearly_data.set_index('year')
sales_2016 = yearly.loc[2016, 'total_sales']
sales_2017 = yearly.loc[2017, 'total_sales']
orders_2016 = yearly.loc[2016, 'order_count']
orders_2017 = yearly.loc[2017, 'order_count']
aov_2016 = yearly.loc[2016, 'avg_order_value']
aov_2017 = yearly.loc[2017, 'avg_order_value']

qty_2016 = price_analysis.iloc[0]['total_quantity']
qty_2017 = price_analysis.iloc[1]['total_quantity']

print(f"\n📉 OVERALL METRICS:")
print(f"  Total Sales Decline: ${sales_2016 - sales_2017:,.0f} (-1.4%)")
print(f"  Order Count Decline: {int(orders_2016 - orders_2017):,} orders (-1.8%)")
print(f"  Quantity Decline: {int(qty_2016 - qty_2017):,} units")

print(f"\n\n🎯 PRIMARY CAUSES OF DECLINE:")
print(f"\n  1️⃣ VOLUME DECLINE (PRIMARY DRIVER)")
print(f"     • Orders decreased by {int(orders_2016 - orders_2017):,} (-1.8%)")
print(f"     • Units sold decreased by {int(qty_2016 - qty_2017):,}")
print(f"     • Average order value INCREASED slightly (+{(aov_2017 - aov_2016)/aov_2016*100:.1f}%)")
print(f"     ➜ Customers are buying more per transaction but fewer transactions overall")

# Find regional issues
print(f"\n  2️⃣ REGIONAL DISPARITIES")
declining_regions = regional_analysis[regional_analysis['sales_change'] < 0]
print(f"     • {len(declining_regions)} out of {len(regional_analysis)} regions declined")
worst = regional_analysis.loc[regional_analysis['sales_change'].idxmin()]
print(f"     • Worst performer: {worst['territoryname']} ({worst['change_%']:+.1f}%)")

# Find product issues
print(f"\n  3️⃣ PRODUCT MIX SHIFT")
declining_products = product_analysis[product_analysis['sales_change'] < 0]
print(f"     • {len(declining_products)} products declined in sales")
print(f"     • Top declining products:")
for idx, (_, row) in enumerate(product_analysis.nlargest(3, 'sales_change', keep='first').iterrows()):
    print(f"       {idx+1}. {row['productname'][:40]}: ${row['sales_change']:,.0f} ({row['change_%']:+.1f}%)")

print(f"\n\n💡 STRATEGIC INSIGHTS:")
print(f"\n  ✓ Not a pricing issue - Avg order value actually INCREASED")
print(f"  ✗ VOLUME PROBLEM - Fewer customers are making purchases")
print(f"  ✗ REGIONAL UNDERPERFORMANCE - Multiple regions showing weakness")
print(f"  ✗ PRODUCT PORTFOLIO CHALLENGE - Key products losing market share")

print(f"\n\n🎬 RECOMMENDED ACTIONS:")
print(f"\n  1. CUSTOMER ACQUISITION: Focus on bringing back lost customers")
print(f"     • 472 fewer orders suggests customer churn or reduced purchase frequency")
print(f"     • Implement loyalty programs to increase repeat purchases")
\n  2. REGIONAL RECOVERY: Investigate declining regions")
print(f"     • Allocate marketing resources to {worst['territoryname']}")
print(f"     • Analyze regional competition and market conditions")
\n  3. PRODUCT REVITALIZATION: Support declining products")
print(f"     • Review top declining products for quality or market fit issues")
print(f"     • Increase promotional activities for key products")
\n  4. SEASONALITY ANALYSIS: Plan for Q4 performance")
print(f"     • Q4 typically strongest - ensure inventory and marketing ready")
\n  5. MARKET ANALYSIS: Understand external factors")
print(f"     • Check for competitive pressure")
print(f"     • Review industry trends and economic conditions")

print("\n" + "="*100)

## 📊 Visual Summary

In [ ]:
# Create a comprehensive summary visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

fig.suptitle('2016 → 2017 Sales Decline: Root Cause Analysis Dashboard', fontsize=18, fontweight='bold', y=0.995)

# 1. Sales comparison
ax1 = fig.add_subplot(gs[0, 0])
years = [2016, 2017]
sales = [9.32, 9.18]
colors_chart = ['#3498db', '#e74c3c']
bars = ax1.bar(years, sales, color=colors_chart, width=0.5)
ax1.set_ylabel('Sales (Millions $)', fontweight='bold')
ax1.set_title('Total Sales', fontweight='bold')
for i, (year, sale) in enumerate(zip(years, sales)):
    ax1.text(year, sale + 0.1, f'${sale}M', ha='center', fontweight='bold')
ax1.set_ylim(8, 10)

# 2. Order count
ax2 = fig.add_subplot(gs[0, 1])
orders = [26953, 26481]
bars = ax2.bar(years, orders, color=colors_chart, width=0.5)
ax2.set_ylabel('Order Count', fontweight='bold')
ax2.set_title('Total Orders (-472)', fontweight='bold')
for i, (year, order) in enumerate(zip(years, orders)):
    ax2.text(year, order + 100, f'{order:,}', ha='center', fontweight='bold')

# 3. Avg order value
ax3 = fig.add_subplot(gs[0, 2])
aov = [346.35, 346.82]
bars = ax3.bar(years, aov, color=colors_chart, width=0.5)
ax3.set_ylabel('AOV ($)', fontweight='bold')
ax3.set_title('Avg Order Value (+$0.47)', fontweight='bold')
for i, (year, val) in enumerate(zip(years, aov)):
    ax3.text(year, val + 1, f'${val:.2f}', ha='center', fontweight='bold')
ax3.set_ylim(340, 352)

# 4. Monthly trend
ax4 = fig.add_subplot(gs[1, :])
months_2016 = sales_pivot[2016]
months_2017 = sales_pivot[2017]
x = np.arange(len(month_names))
ax4.plot(x, months_2016/1e6, marker='o', label='2016', linewidth=2, color='#3498db')
ax4.plot(x, months_2017/1e6, marker='s', label='2017', linewidth=2, color='#e74c3c')
ax4.fill_between(x, months_2016/1e6, months_2017/1e6, alpha=0.2, color='#e67e22')
ax4.set_xticks(x)
ax4.set_xticklabels(month_names)
ax4.set_ylabel('Monthly Sales (Millions $)', fontweight='bold')
ax4.set_title('Monthly Sales Trend',fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Regional performance
ax5 = fig.add_subplot(gs[2, 0])
regions_short = [r[:15] + '...' if len(r) > 15 else r for r in regional_analysis['territoryname']]
colors_regional = ['#2ecc71' if x > 0 else '#e74c3c' for x in regional_analysis['change_%']]
ax5.barh(regions_short, regional_analysis['change_%'], color=colors_regional)
ax5.set_xlabel('% Change', fontweight='bold')
ax5.set_title('Regional Performance', fontweight='bold')
ax5.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

# 6. Top declining products
ax6 = fig.add_subplot(gs[2, 1])
top_decline = product_analysis.nlargest(5, 'sales_change', keep='first').sort_values('sales_change')
product_names_short = [p[:20] + '...' if len(p) > 20 else p for p in top_decline['productname']]
ax6.barh(product_names_short, top_decline['change_%'], color='#e74c3c', alpha=0.7)
ax6.set_xlabel('% Change', fontweight='bold')
ax6.set_title('Top 5 Declining Products', fontweight='bold')

# 7. Key metrics box
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')
summary_text = f"""KEY FINDINGS
━━━━━━━━━━━━━━━━━━━━━━

Sales Decline: -1.4%

Primary Cause:
VOLUME DECLINE

Orders Down: -472 (-1.8%)

Impact by Region:
{len(declining_regions)} declining
{len(regional_analysis) - len(declining_regions)} growing

Product Issues:
{len(declining_products)} declining
"""
ax7.text(0.1, 0.9, summary_text, transform=ax7.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig('reports/summary_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Comprehensive dashboard saved: reports/summary_dashboard.png")

## 📝 Conclusion

In [ ]:
print("\n" + "="*100)
print("🎯 EXECUTIVE SUMMARY - THE STORY OF SALES DECLINE")
print("="*100)

conclusion = """
The 1.4% sales decline from 2016 ($9.32M) to 2017 ($9.18M) reveals a clear underlying narrative:

🔴 THE PRIMARY ISSUE: Volume Decline
   • Order count dropped by 472 transactions (-1.8%)
   • Total units sold decreased significantly
   • However, average order value INCREASED (+$0.47, +0.1%)
   → Conclusion: Fewer people buying, but buying more per transaction

⚠️  SECONDARY ISSUES:
   
   1. Regional Weakness
      • Several key territories showed decline
      • {worst_region['territoryname']} was particularly weak
      • Geographic disparities need urgent attention
   
   2. Product Portfolio Challenges  
      • Multiple high-volume products lost sales
      • While some new products gained traction
      • Portfolio shift indicates market dynamics changing
   
   3. Customer Acquisition/Retention
      • Order decline suggests fewer unique transactions
      • Possible customer churn or reduced purchase frequency
      • Repeat customer patterns need investigation

✅ POSITIVE SIGNALS:
   • Average order value increased - high-value customers still engaged
   • Several regions and products showed growth
   • Some customer segments strengthening
   • Core business fundamentals remained stable

💼 BUSINESS IMPLICATIONS:
   
   The -1.4% decline is MANAGEABLE but signals:  
   1. Loss of transaction volume (not profitability)
   2. Market becoming more selective/quality-focused
   3. Need for customer acquisition initiatives
   4. Regional support needed in struggling territories
   5. Product portfolio rebalancing underway

🚀 PATH TO RECOVERY:
   1. Focus on customer acquisition - restore transaction count
   2. Increase repeat purchase rates - leverage loyalty programs
   3. Support regional growth - especially high-performing territories
   4. Optimize product mix - support growing products, revitalize declining ones
   5. Monitor market trends - external factors may be at play
"""

print(conclusion)
print("="*100)